In [1]:

from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
import pandas as pd
import pickle
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['font.serif'] = 'Times New Roman'
mpl.rcParams['font.size'] = 10

In [2]:
import time
import sys
import os,glob
from collections import deque
from typing import Dict, Tuple

import gymnasium as gym
import numpy as np
import torch
from torch import Tensor

from sample_factory.algo.learning.learner import Learner
from sample_factory.algo.sampling.batched_sampling import preprocess_actions
from sample_factory.algo.utils.action_distributions import argmax_actions
from sample_factory.algo.utils.env_info import extract_env_info
from sample_factory.algo.utils.make_env import make_env_func_batched
from sample_factory.algo.utils.misc import ExperimentStatus
from sample_factory.algo.utils.rl_utils import make_dones, prepare_and_normalize_obs
from sample_factory.algo.utils.tensor_utils import unsqueeze_tensor
from sample_factory.cfg.arguments import load_from_checkpoint
# from sample_factory.huggingface.huggingface_utils import generate_model_card, generate_replay_video, push_to_hf
from sample_factory.model.actor_critic import create_actor_critic
from sample_factory.model.model_utils import get_rnn_size
from sample_factory.utils.attr_dict import AttrDict
from sample_factory.utils.typing import Config, StatusCode
from sample_factory.utils.utils import debug_log_every_n, experiment_dir, log

# ---------------------------------------------------------------------------
# logging helpers (put near the top of the file, after imports)
# ---------------------------------------------------------------------------
import datetime, pathlib, json, pandas as pd, torch

def _ensure_parent(path: pathlib.Path):
    path.parent.mkdir(parents=True, exist_ok=True)

/home/fr/fr_lr554/.conda/envs/env/lib/python3.10/site-packages/deepmind_lab/__init__.py:26: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
import sys
from multiprocessing.context import BaseContext
from typing import Optional

from tensorboardX import SummaryWriter

from sample_factory.algo.runners.runner import AlgoObserver, Runner
from sample_factory.algo.utils.context import global_model_factory
from sample_factory.algo.utils.misc import ExperimentStatus
from sample_factory.algo.utils.multiprocessing_utils import get_mp_ctx
from sample_factory.cfg.arguments import parse_full_cfg, parse_sf_args
from sample_factory.envs.env_utils import register_env
from sample_factory.train import make_runner
from sample_factory.utils.typing import Config, Env, PolicyID
from sample_factory.utils.utils import experiment_dir

# from sf_workingdir_lilly.dmlab.dmlab_env import (
#     DMLAB_ENVS,
#     dmlab_extra_episodic_stats_processing,
#     dmlab_extra_summaries,
#     list_all_levels_for_experiment,
#     make_dmlab_env,
# )
from sf_workingdir_lilly.dmlab.dmlab_level_cache import DmlabLevelCaches, make_dmlab_caches
# from sf_examples.dmlab.dmlab_model import make_dmlab_encoder
from sf_workingdir_lilly.dmlab.custom_core import make_hipposlam_core
from sf_workingdir_lilly.dmlab.custom_encoder import make_hipposlam_encoder
from sf_workingdir_lilly.dmlab.dmlab_params import add_dmlab_env_args, dmlab_override_defaults
from sf_workingdir_lilly.dmlab.custom_params import add_hipposlam_env_args, hipposlam_override_defaults


class DmlabEnvWithCache:
    def __init__(self, level_caches: Optional[DmlabLevelCaches] = None):
        self.caches = level_caches

    def make_env(self, env_name, cfg, env_config, render_mode) -> Env:
        return make_dmlab_env(env_name, cfg, env_config, render_mode, self.caches)


def register_dmlab_envs(level_caches: Optional[DmlabLevelCaches] = None):
    env_factory = DmlabEnvWithCache(level_caches)
    for env in DMLAB_ENVS:
        register_env(env.name, env_factory.make_env)


def register_dmlab_components(level_caches: Optional[DmlabLevelCaches] = None):
    # register_dmlab_envs(level_caches)
    global_model_factory().register_encoder_factory(make_hipposlam_encoder)
    global_model_factory().register_model_core_factory(make_hipposlam_core)


class DmlabExtraSummariesObserver(AlgoObserver):
    def extra_summaries(self, runner: Runner, policy_id: PolicyID, writer: SummaryWriter, env_steps: int) -> None:
        dmlab_extra_summaries(runner, policy_id, writer, env_steps)


def register_msg_handlers(cfg: Config, runner: Runner):
    if cfg.env == "dmlab_30":
        # extra functions to calculate human-normalized score etc.
        runner.register_episodic_stats_handler(dmlab_extra_episodic_stats_processing)
        runner.register_observer(DmlabExtraSummariesObserver())


def initialize_level_cache(cfg: Config, mp_ctx: BaseContext) -> Optional[DmlabLevelCaches]:
    if not cfg.dmlab_use_level_cache:
        return None

    env_name = cfg.env
    num_policies = cfg.num_policies if hasattr(cfg, "num_policies") else 1
    all_levels = list_all_levels_for_experiment(env_name)
    level_cache_dir = cfg.dmlab_level_cache_path
    caches = make_dmlab_caches(experiment_dir(cfg), all_levels, num_policies, level_cache_dir, mp_ctx)
    return caches


def parse_dmlab_args(argv=None, evaluation=False):
    parser, cfg = parse_sf_args(argv, evaluation=evaluation)
    add_hipposlam_env_args(parser)
    add_dmlab_env_args(parser)
    hipposlam_override_defaults(parser)
    cfg = parse_full_cfg(parser, argv)
    return cfg



In [4]:

REDUCED_ACTION_SET = (
    (0, 0, 0, 1, 0, 0, 0),  # Forward
    # (0, 0, 0, -1, 0, 0, 0),  # Backward
    (0, 0, -1, 0, 0, 0, 0),  # Strafe Left
    (0, 0, 1, 0, 0, 0, 0),  # Strafe Right
    # (-20, 0, 0, 0, 0, 0, 0),  # Look Left
    # (20, 0, 0, 0, 0, 0, 0),  # Look Right
    (-20, 0, 0, 1, 0, 0, 0),  # Look Left + Forward
    (20, 0, 0, 1, 0, 0, 0),  # Look Right + Forward
    # (0, 0, 0, 0, 1, 0, 0),  # Fire.
)
action_space = gym.spaces.Discrete(len(REDUCED_ACTION_SET))
observation_space = gym.spaces.Dict(
    obs=gym.spaces.Box(low=0, high=255, shape=[4, 72, 96], dtype=np.uint8)
)

In [5]:
%cd /work/classic/fr_lr554-TrainSpace/spectral_radius

/work/classic/fr_lr554-TrainSpace/spectral_radius


In [6]:
trained_lora = dict()
weight_trials = [1, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 17, 18, 22, 23, 24, 27, 28, 29, 30, 33, 35, 36, 38, 39, 40, 41, 42, 43, 45, 47, 49]
weight_trials_unordered = [14,39,23,45,8,22,17,15,38,4,11,1,9,3,18,33,6,12,49,43,7,41,35,5,40,42,27,36,13,24,28,30,47,29]
seeds = [1111,2222,3333, 4444,5555]

simulation = 'RNNRandomLORA47'

In [7]:
ids = [f"{i:02d}" for i in range(170)]
print(ids)

['00', '01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '154', '155', '156', 

In [8]:
# the final dataframe will have the structure dict with weight trials, dict with seeds, and then the two lora components in there
all_lora = dict()

In [9]:
for weight_i, t in enumerate(weight_trials_unordered):
    df_lora_current_weight_trial = {1111:{'lr_column':None, 'lr_row':None}, 
                    2222:{'lr_column':None, 'lr_row':None}, 
                    3333:{'lr_column':None, 'lr_row':None},
                    4444:{'lr_column':None, 'lr_row':None},
                    5555:{'lr_column':None, 'lr_row':None}}
    for seed_i, seed in enumerate(seeds):
        cfg_filename=f'/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/{simulation}_/{ids[weight_i+len(weight_trials_unordered)*seed_i]}_{simulation}_see_{seed}_w.tri_{t}/config.json'
        with open(cfg_filename, "r") as json_file:
            json_params = json.load(json_file)
            log.warning("Loading existing experiment configuration from %s", cfg_filename)
            loaded_cfg = AttrDict(json_params)

        cfg=loaded_cfg

        mapname="openfield_map2_fixed_loc3"
        expname=f'hipposlam/{simulation}_/{ids[weight_i+len(weight_trials_unordered)*seed_i]}_{simulation}_see_{seed}_w.tri_{t}'

        cli = [
            "--algo", "APPO",
            "--env", mapname ,         # pick any DM‑Lab level you have
            "--experiment", expname,
            "--encoder_load_path","/home/fr/fr_lr554/best_000025288_203030528_reward_94.185.pth",
            "--train_dir", "/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir", # anything writable
            "--max_num_frames", "50000",          # short rollout for the test
            "--num_envs", "8",
            "--dmlab_level_cache_path","./.dmlab_cache",
            "--load_checkpoint_kind","latest",
            "--use_jit","False",
            "--with_pos_obs","True",
            "--depth_sensor", "True",
            "--no_render",        # <-- skip human window; avoid X11 on servers
        ]

        cli_dict={
        'algo': 'APPO',
        'env': mapname,
        'experiment': expname,
        'encoder_load_path': '/home/fr/fr_lr554/best_000025288_203030528_reward_94.185.pth',
        'train_dir': '/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir',
        'max_num_frames': '50000',
        'num_envs': '8',
        'dmlab_level_cache_path': './.dmlab_cache',
        'load_checkpoint_kind': 'latest',
        'no_render': True,
        'use_jit': False,
        'with_pos_obs': True,
        "depth_sensor": True,
        }
        register_dmlab_components()
        cfg = parse_dmlab_args(evaluation=True, argv=cli)

        cfg.train_dir = "/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir"
        cfg.experiment = expname

        pattern = os.path.join(cfg.train_dir, cfg.experiment, "**", "*.pth")
        ckpts = glob.glob(pattern, recursive=True)
        assert ckpts, f"No checkpoints found under {pattern}"
        ckpt_path = max(ckpts, key=os.path.getmtime)
        print("Using checkpoint:", ckpt_path)

        # tweak whatever you like *after* parsing
        # cfg.with_pos_obs = True
        cfg.cli_args=cli_dict
        # status = enjoy(cfg)


        verbose = False

        cfg = load_from_checkpoint(cfg)

        eval_env_frameskip: int = cfg.env_frameskip if cfg.eval_env_frameskip is None else cfg.eval_env_frameskip
        assert (
            cfg.env_frameskip % eval_env_frameskip == 0
        ), f"{cfg.env_frameskip=} must be divisible by {eval_env_frameskip=}"
        render_action_repeat: int = cfg.env_frameskip // eval_env_frameskip
        cfg.env_frameskip = cfg.eval_env_frameskip = eval_env_frameskip
        log.debug(f"Using frameskip {cfg.env_frameskip} and {render_action_repeat=} for evaluation")

        cfg.num_envs = 1

        render_mode = "human"
        if cfg.save_video:
            render_mode = "rgb_array"
        elif cfg.no_render:
            render_mode = None

        # env = make_env_func_batched(
        #     cfg, env_config=AttrDict(worker_index=0, vector_index=0, env_id=0), render_mode=render_mode
        # )
        # env_info = extract_env_info(env, cfg)
        # if hasattr(env.unwrapped, "reset_on_init"):
        #     # reset call ruins the demo recording for VizDoom
        #     env.unwrapped.reset_on_init = False
        # log.info(env.action_space)
        actor_critic = create_actor_critic(cfg, observation_space, action_space)


        actor_critic.eval()
        policy_id = cfg.policy_index
        # log.info(policy_id)
        name_prefix = dict(latest="checkpoint", best="best")[cfg.load_checkpoint_kind]
        # log.info(Learner.checkpoint_dir(cfg, policy_id))
        # checkpoints = Learner.get_checkpoints(Learner.checkpoint_dir(cfg, policy_id), f"{name_prefix}_*")
        # checkpoint_dict = Learner.load_checkpoint(str(pth_name), device)
        checkpoint_dict = torch.load(str(ckpt_path), 'cpu', weights_only=False)
        actor_critic.load_state_dict(checkpoint_dict["model"])




        print('#######################################')
        print(t, seed)
        df_lora_current_weight_trial[seed]['lr_column'] =  actor_critic.core.rnn.lr_column.detach().cpu().numpy()
        df_lora_current_weight_trial[seed]['lr_row'] =  actor_critic.core.rnn.lr_row.detach().cpu().numpy()
        print(actor_critic.core.rnn.lr_column)
        print(actor_critic.core.rnn.lr_row)
        #break
    all_lora[t] = df_lora_current_weight_trial
    #break
        

[2026-02-04 14:44:20,543][411013] Loading existing experiment configuration from /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/00_RNNRandomLORA47_see_1111_w.tri_14/config.json
[2026-02-04 14:44:20,544][411013] register_encoder_factory: <function make_hipposlam_encoder at 0x7f103ac29750>
[2026-02-04 14:44:20,545][411013] register_model_core_factory: <function make_hipposlam_core at 0x7f1047158b80>
[2026-02-04 14:44:20,587][411013] Loading existing experiment configuration from /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/00_RNNRandomLORA47_see_1111_w.tri_14/config.json
[2026-02-04 14:44:20,588][411013] Overriding arg 'experiment' with value 'hipposlam/RNNRandomLORA47_/00_RNNRandomLORA47_see_1111_w.tri_14' passed from command line
[2026-02-04 14:44:20,588][411013] Overriding arg 'train_dir' with value '/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir' passed from command line
[2026-02-04 14:44:2

Using checkpoint: /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/00_RNNRandomLORA47_see_1111_w.tri_14/checkpoint_p3/checkpoint_000010706_87703552.pth
#######################################
14 1111
Parameter containing:
tensor([[-0.0467],
        [-0.0050],
        [-0.0070],
        ...,
        [-0.0626],
        [-0.0231],
        [ 0.0310]], requires_grad=True)
Parameter containing:
tensor([[-0.0611,  0.0128,  0.0139,  ..., -0.0521,  0.0134,  0.0349]],
       requires_grad=True)


[2026-02-04 14:44:20,794][411013] Loading existing experiment configuration from /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/34_RNNRandomLORA47_see_2222_w.tri_14/config.json
[2026-02-04 14:44:20,794][411013] Overriding arg 'experiment' with value 'hipposlam/RNNRandomLORA47_/34_RNNRandomLORA47_see_2222_w.tri_14' passed from command line
[2026-02-04 14:44:20,794][411013] Overriding arg 'train_dir' with value '/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir' passed from command line
[2026-02-04 14:44:20,794][411013] Overriding arg 'dmlab_level_cache_path' with value './.dmlab_cache' passed from command line
[2026-02-04 14:44:20,795][411013] Overriding arg 'use_jit' with value False passed from command line
[2026-02-04 14:44:20,795][411013] Overriding arg 'with_pos_obs' with value True passed from command line
[2026-02-04 14:44:20,795][411013] Adding new argument 'fps'=0 that is not in the saved config file!
[2026-02-04 14:44:20,795][4

Using checkpoint: /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/34_RNNRandomLORA47_see_2222_w.tri_14/checkpoint_p0/checkpoint_000009149_74948608.pth
#######################################
14 2222
Parameter containing:
tensor([[ 0.0024],
        [-0.0005],
        [ 0.0096],
        ...,
        [-0.0420],
        [ 0.0004],
        [ 0.0304]], requires_grad=True)
Parameter containing:
tensor([[ 0.0170,  0.0348,  0.0486,  ..., -0.0097,  0.0315,  0.0061]],
       requires_grad=True)
Using checkpoint: /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/68_RNNRandomLORA47_see_3333_w.tri_14/checkpoint_p2/checkpoint_000012180_99778560.pth


[2026-02-04 14:44:21,033][411013] Loading existing experiment configuration from /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/102_RNNRandomLORA47_see_4444_w.tri_14/config.json
[2026-02-04 14:44:21,033][411013] register_encoder_factory: <function make_hipposlam_encoder at 0x7f103ac29750>
[2026-02-04 14:44:21,033][411013] register_model_core_factory: <function make_hipposlam_core at 0x7f1047158b80>
[2026-02-04 14:44:21,051][411013] Loading existing experiment configuration from /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/102_RNNRandomLORA47_see_4444_w.tri_14/config.json
[2026-02-04 14:44:21,051][411013] Overriding arg 'experiment' with value 'hipposlam/RNNRandomLORA47_/102_RNNRandomLORA47_see_4444_w.tri_14' passed from command line
[2026-02-04 14:44:21,051][411013] Overriding arg 'train_dir' with value '/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir' passed from command line
[2026-02-04 14:4

...

#######################################
47 4444
Parameter containing:
tensor([[-0.0352],
        [ 0.0064],
        [ 0.0335],
        ...,
        [-0.0050],
        [-0.0193],
        [ 0.0087]], requires_grad=True)
Parameter containing:
tensor([[-0.0368, -0.0290,  0.0120,  ..., -0.0200, -0.0006,  0.0137]],
       requires_grad=True)
Using checkpoint: /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/168_RNNRandomLORA47_see_5555_w.tri_47/checkpoint_p1/checkpoint_000009891_81027072.pth
#######################################
47 5555
Parameter containing:
tensor([[ 0.0058],
        [ 0.0387],
        [ 0.0127],
        ...,
        [ 0.0223],
        [ 0.0028],
        [-0.0085]], requires_grad=True)
Parameter containing:
tensor([[ 0.0241,  0.0368, -0.0128,  ...,  0.0011,  0.0189,  0.0052]],
       requires_grad=True)
Using checkpoint: /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/33_RNNRandomLORA47_see_1111_

[2026-02-04 14:44:38,995][411013] Loading existing experiment configuration from /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/67_RNNRandomLORA47_see_2222_w.tri_29/config.json
[2026-02-04 14:44:38,995][411013] register_encoder_factory: <function make_hipposlam_encoder at 0x7f103ac29750>
[2026-02-04 14:44:38,995][411013] register_model_core_factory: <function make_hipposlam_core at 0x7f1047158b80>
[2026-02-04 14:44:39,016][411013] Loading existing experiment configuration from /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/67_RNNRandomLORA47_see_2222_w.tri_29/config.json
[2026-02-04 14:44:39,017][411013] Overriding arg 'experiment' with value 'hipposlam/RNNRandomLORA47_/67_RNNRandomLORA47_see_2222_w.tri_29' passed from command line
[2026-02-04 14:44:39,017][411013] Overriding arg 'train_dir' with value '/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir' passed from command line
[2026-02-04 14:44:3

#######################################
29 1111
Parameter containing:
tensor([[-0.0285],
        [-0.0090],
        [-0.0179],
        ...,
        [ 0.0069],
        [-0.0336],
        [-0.0008]], requires_grad=True)
Parameter containing:
tensor([[ 0.0008,  0.0478,  0.0492,  ...,  0.0156, -0.0208,  0.0527]],
       requires_grad=True)
Using checkpoint: /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/67_RNNRandomLORA47_see_2222_w.tri_29/checkpoint_p1/checkpoint_000012496_102367232.pth
#######################################
29 2222
Parameter containing:
tensor([[-0.0028],
        [ 0.0012],
        [-0.0279],
        ...,
        [ 0.0043],
        [-0.0428],
        [ 0.0227]], requires_grad=True)
Parameter containing:
tensor([[ 0.0088,  0.0092,  0.0184,  ..., -0.0172,  0.0021, -0.0206]],
       requires_grad=True)
Using checkpoint: /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/101_RNNRandomLORA47_see_3333

[2026-02-04 14:44:39,206][411013] Loading existing experiment configuration from /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/135_RNNRandomLORA47_see_4444_w.tri_29/config.json
[2026-02-04 14:44:39,207][411013] register_encoder_factory: <function make_hipposlam_encoder at 0x7f103ac29750>
[2026-02-04 14:44:39,207][411013] register_model_core_factory: <function make_hipposlam_core at 0x7f1047158b80>
[2026-02-04 14:44:39,224][411013] Loading existing experiment configuration from /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/135_RNNRandomLORA47_see_4444_w.tri_29/config.json
[2026-02-04 14:44:39,225][411013] Overriding arg 'experiment' with value 'hipposlam/RNNRandomLORA47_/135_RNNRandomLORA47_see_4444_w.tri_29' passed from command line
[2026-02-04 14:44:39,225][411013] Overriding arg 'train_dir' with value '/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir' passed from command line
[2026-02-04 14:4

#######################################
29 3333
Parameter containing:
tensor([[-4.4184e-02],
        [ 7.3346e-07],
        [ 1.8974e-02],
        ...,
        [ 1.3799e-03],
        [ 1.9286e-02],
        [-1.9925e-02]], requires_grad=True)
Parameter containing:
tensor([[-0.0055, -0.0188,  0.0172,  ...,  0.0176, -0.0027,  0.0228]],
       requires_grad=True)
Using checkpoint: /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/135_RNNRandomLORA47_see_4444_w.tri_29/checkpoint_p0/checkpoint_000008978_73547776.pth
#######################################
29 4444
Parameter containing:
tensor([[-0.0171],
        [ 0.0019],
        [ 0.0115],
        ...,
        [ 0.0084],
        [-0.0122],
        [-0.0013]], requires_grad=True)
Parameter containing:
tensor([[-0.0282, -0.0191,  0.0582,  ...,  0.0260,  0.0046,  0.0439]],
       requires_grad=True)
Using checkpoint: /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/169_

In [15]:
cfg.pixel_format


'CHW'

In [12]:
ckpt_path

'/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/00_RNNRandomLORA47_see_1111_w.tri_14/checkpoint_p3/checkpoint_000010706_87703552.pth'

In [10]:
print(all_lora.keys())
print(len(all_lora.keys()))
print(all_lora[41][1111])

dict_keys([14, 39, 23, 45, 8, 22, 17, 15, 38, 4, 11, 1, 9, 3, 18, 33, 6, 12, 49, 43, 7, 41, 35, 5, 40, 42, 27, 36, 13, 24, 28, 30, 47, 29])
34
{'lr_column': array([[-0.00679616],
       [ 0.01847937],
       [-0.00150866],
       ...,
       [-0.02427179],
       [ 0.00443497],
       [ 0.02775938]], shape=(1136, 1), dtype=float32), 'lr_row': array([[ 0.00823138,  0.01361715,  0.034117  , ..., -0.01573449,
         0.02793941,  0.04772   ]], shape=(1, 1136), dtype=float32)}


In [59]:
print("train_dir:", cfg.train_dir)
print("experiment:", cfg.experiment)


train_dir: /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir
experiment: hipposlam/RNNRandomLORA47_/00_RNNRandomLORA47_see_1111_w.tri_14


In [60]:
import glob, os

pattern = os.path.join(cfg.train_dir, cfg.experiment, "**", "*.pth")
ckpts = glob.glob(pattern, recursive=True)
print("Found ckpts:", ckpts[:5], " ... total:", len(ckpts))
assert len(ckpts) > 0, "No checkpoint files found for this experiment!"


Found ckpts: ['/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/00_RNNRandomLORA47_see_1111_w.tri_14/checkpoint_p0/checkpoint_000012565_102932480.pth', '/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/00_RNNRandomLORA47_see_1111_w.tri_14/checkpoint_p0/checkpoint_000012573_102998016.pth', '/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/00_RNNRandomLORA47_see_1111_w.tri_14/checkpoint_p0/checkpoint_000012598_103202816.pth', '/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/00_RNNRandomLORA47_see_1111_w.tri_14/checkpoint_p0/checkpoint_000012609_103292928.pth', '/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA47_/00_RNNRandomLORA47_see_1111_w.tri_14/checkpoint_p0/checkpoint_000012584_103088128.pth']  ... total: 188


In [52]:
trained_lora[5555]= {'lr_column': actor_critic.core.rnn.lr_column.detach().cpu().numpy(),
                    'lr_row': actor_critic.core.rnn.lr_row.detach().cpu().numpy()}

In [17]:
#print(trained_lora[5555])
print(all_lora[41][1111]['lr_column'])

[[-0.00679616]
 [ 0.01847937]
 [-0.00150866]
 ...
 [-0.02427179]
 [ 0.00443497]
 [ 0.02775938]]


In [11]:
# save the extracted data
with open("/home/fr/fr_lr554/samplefactory/sample-factory/sf_workingdir_lilly/dmlab/analysis/data/sim47_lora.pkl", "wb") as f:
    pickle.dump(all_lora, f)

In [ ]:
########################## STOP ##########################

In [ ]:
import torch.nn as nn

In [ ]:
print(type(actor_critic.core))
print(isinstance(actor_critic.core, nn.Module))

print("children:", list(actor_critic.core.named_children()))
print("modules :", list(actor_critic.core.named_modules())[:10])
print("params  :", list(actor_critic.core.named_parameters())[:10])

<class 'sf_examples.dmlab.Hipposlam_model.SimpleSequenceWithBypassCore'>
True
children: []
modules : [('', SimpleSequenceWithBypassCore())]
params  : []


In [ ]:
actor_critic.core.named_parameters()

dict_keys([])

In [ ]:
dict_keys(['', 'obs_normalizer', 'obs_normalizer.running_mean_std', 'obs_normalizer.running_mean_std.running_mean_std', 'obs_normalizer.running_mean_std.running_mean_std.obs', 'returns_normalizer', 'encoder', 'encoder.depth_encoder', 'encoder.depth_encoder.downsample', 'encoder.basic_encoder', 'encoder.basic_encoder.conv_head', 'encoder.basic_encoder.conv_head.0', 'encoder.basic_encoder.conv_head.1', 'encoder.basic_encoder.conv_head.2', 'encoder.basic_encoder.conv_head.2.res_block_core', 'encoder.basic_encoder.conv_head.2.res_block_core.0', 'encoder.basic_encoder.conv_head.2.res_block_core.1', 'encoder.basic_encoder.conv_head.2.res_block_core.2', 'encoder.basic_encoder.conv_head.2.res_block_core.3', 'encoder.basic_encoder.conv_head.3', 'encoder.basic_encoder.conv_head.3.res_block_core', 'encoder.basic_encoder.conv_head.3.res_block_core.0', 'encoder.basic_encoder.conv_head.3.res_block_core.1', 'encoder.basic_encoder.conv_head.3.res_block_core.2', 'encoder.basic_encoder.conv_head.3.res_block_core.3', 'encoder.basic_encoder.conv_head.4', 'encoder.basic_encoder.conv_head.5', 'encoder.basic_encoder.conv_head.6', 'encoder.basic_encoder.conv_head.6.res_block_core', 'encoder.basic_encoder.conv_head.6.res_block_core.0', 'encoder.basic_encoder.conv_head.6.res_block_core.1', 'encoder.basic_encoder.conv_head.6.res_block_core.2', 'encoder.basic_encoder.conv_head.6.res_block_core.3', 'encoder.basic_encoder.conv_head.7', 'encoder.basic_encoder.conv_head.7.res_block_core', 'encoder.basic_encoder.conv_head.7.res_block_core.0', 'encoder.basic_encoder.conv_head.7.res_block_core.1', 'encoder.basic_encoder.conv_head.7.res_block_core.2', 'encoder.basic_encoder.conv_head.7.res_block_core.3', 'encoder.basic_encoder.conv_head.8', 'encoder.basic_encoder.conv_head.9', 'encoder.basic_encoder.conv_head.10', 'encoder.basic_encoder.conv_head.10.res_block_core', 'encoder.basic_encoder.conv_head.10.res_block_core.0', 'encoder.basic_encoder.conv_head.10.res_block_core.1', 'encoder.basic_encoder.conv_head.10.res_block_core.2', 'encoder.basic_encoder.conv_head.10.res_block_core.3', 'encoder.basic_encoder.conv_head.11', 'encoder.basic_encoder.conv_head.11.res_block_core', 'encoder.basic_encoder.conv_head.11.res_block_core.0', 'encoder.basic_encoder.conv_head.11.res_block_core.1', 'encoder.basic_encoder.conv_head.11.res_block_core.2', 'encoder.basic_encoder.conv_head.11.res_block_core.3', 'encoder.basic_encoder.conv_head.12', 'encoder.basic_encoder.mlp_layers', 'encoder.basic_encoder.mlp_layers.0', 'encoder.DG_projection', 'encoder.DG_projection.linear', 'encoder.DG_projection.batchnorm1d', 'encoder.DG_projection.activation', 'core', 'decoder', 'decoder.mlp', 'decoder.mlp.0', 'decoder.mlp.1', 'decoder.mlp.2', 'critic_linear', 'action_parameterization', 'action_parameterization.distribution_linear'])

In [ ]:
[x for x in dict(actor_critic.named_modules()).keys() if x.startswith('encoder.DG')]